# Ingestion coverage — code-review-graph on `spring-petclinic`

Spring PetClinic (Java + Spring Boot, Maven). Fills the team's `Ingestion_Coverage.xlsx` for this project.
**Functions expected** come from javac's own parser (`ref_java.py` → `java-ref/ListMethods.java`); needs a JDK.

Every run is a new iteration in `results/spring-petclinic/runs/iter-NNN_<date>_<time>/` and adds
lines to `results/spring-petclinic/runs.csv`; nothing from an earlier run is overwritten.

Run: `./run.sh spring-petclinic` (from `notebooks/`), or Run All here and Save As into the iteration folder.

In [6]:
APP = 'spring-petclinic'   # this notebook's project (folder under apps/)
RUN = ''        # iteration folder name; empty = next free iter-NNN (run.sh sets it)

In [7]:
import sys; sys.path.insert(0, '.')
import pandas as pd
import coverage_lab as C
import ref_java
REFERENCES = [ref_java]   # this project's languages that get a functions-expected count
pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 120)
RUN = RUN or C.next_run_id(APP)
OUT = C.run_dir(APP, RUN)   # results/<APP>/runs/<RUN>/
stamp = C.run_stamp(APP)
{**stamp, 'run': RUN}

$ git rev-parse --short HEAD 
$ /Users/thiru/.local/bin/code-review-graph --version 


{'app': 'spring-petclinic',
 'app_commit': '500158f',
 'tool': 'code-review-graph 2.3.8',
 'run_at': '2026-11-17T20:08:21',
 'run': 'iter-002_2026-11-17_2008'}

## 1. Build the graph exactly as Synthetix does
`CRG_DATA_DIR=<dir> code-review-graph build --repo <root> -q`, after removing the repo from the tool's registry.

In [8]:
db = C.build(APP, OUT)
db

$ /Users/thiru/.local/bin/code-review-graph repos 
$ /Users/thiru/.local/bin/code-review-graph build --repo /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/spring-petclinic -q (env {'CRG_DATA_DIR': '/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/spring-petclinic/runs/iter-002_2026-11-17_2008/crg'})


PosixPath('/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/spring-petclinic/runs/iter-002_2026-11-17_2008/crg/graph.db')

## 2. Sheet 1 — one row per code file
Read? / Why come from the tool's own filter and a re-parse with the same grammar.

In [9]:
rows, funcs, left_out = C.coverage(APP, db, REFERENCES)
sheet1 = pd.DataFrame([{k: v for k, v in r.__dict__.items() if k != 'expected'} for r in rows])
sheet1.read.value_counts()

$ git ls-files 
$ java ListMethods.java spring-petclinic (50 files on stdin)


read
Fully read     61
Not read        8
Partly read     1
Name: count, dtype: int64

### Files left out of sheet 1 (not code)
No language in the tool and not a programming-language extension: docs, licences, images, Dockerfiles, config.

In [10]:
print(len(left_out), 'files left out')
pd.Series([p.rsplit('.', 1)[-1] if '.' in p.rsplit('/', 1)[-1] else '(no extension)' for p in left_out]).value_counts()

62 files left out


properties        13
html              12
txt                4
png                4
scss               4
xml                3
svg                3
gradle             2
eot                2
ttf                2
woff               2
(no extension)     1
json               1
editorconfig       1
gitattributes      1
gitignore          1
md                 1
jar                1
bat                1
cmd                1
css                1
jmx                1
Name: count, dtype: int64

In [11]:
sheet1[sheet1.read != 'Fully read'][['file', 'language', 'lines', 'read', 'why', 'lines_read', 'notes']]

,file,language,lines,read,why,lines_read,notes
0,.github/dco.yml,yaml,2,Not read,Language dropped,0,"passes the filter, parser returns nothing (parser.py:2674-2697)"
1,.github/workflows/deploy-and-test-cluster.yml,yaml,31,Not read,Language dropped,0,"passes the filter, parser returns nothing (parser.py:2674-2697)"
2,.github/workflows/gradle-build.yml,yaml,31,Not read,Language dropped,0,"passes the filter, parser returns nothing (parser.py:2674-2697)"
3,.github/workflows/maven-build.yml,yaml,29,Not read,Language dropped,0,"passes the filter, parser returns nothing (parser.py:2674-2697)"
4,.gitpod.yml,yaml,9,Not read,Language dropped,0,"passes the filter, parser returns nothing (parser.py:2674-2697)"
5,docker-compose.yml,yaml,21,Not read,Language dropped,0,"passes the filter, parser returns nothing (parser.py:2674-2697)"
6,gradlew,bash,248,Partly read,Parse error,247,1 lines inside syntax errors: 178
7,k8s/db.yml,yaml,73,Not read,Language dropped,0,"passes the filter, parser returns nothing (parser.py:2674-2697)"
8,k8s/petclinic.yml,yaml,64,Not read,Language dropped,0,"passes the filter, parser returns nothing (parser.py:2674-2697)"


## 3. Functions: expected (language's own parser) vs found (graph.db)

In [12]:
matched = C.match_functions(rows, funcs)
matched.groupby(['language', 'status']).size()

language  status                                         
java      found                                              192
          lost: same name stored once (graph.py:79, :243)      3
dtype: int64

In [13]:
matched[matched.status != 'found']

,language,file,function,line,status
20,java,src/main/java/org/springframework/samples/petclinic/owner/Owner.java,getPet,120,"lost: same name stored once (graph.py:79, :243)"
21,java,src/main/java/org/springframework/samples/petclinic/owner/Owner.java,getPet,129,"lost: same name stored once (graph.py:79, :243)"
90,java,src/main/java/org/springframework/samples/petclinic/vet/VetRepository.java,findAll,44,"lost: same name stored once (graph.py:79, :243)"


### Files the reference parser could not fully check
Listed, never silently dropped (a syntax error, or code the reference cannot parse).

In [14]:
C.could_not_check(rows)

,language,file,reason


### Unnamed functions (callbacks, lambdas) — not counted as expected
Shown per file with the lines they cover. Their code is still walked by the tool; only a separate row is missing.

In [15]:
cb = C.callbacks_table(rows)
print(cb.callbacks.sum(), 'unnamed functions,', cb.lines.sum(), 'lines')
cb.sort_values('callbacks', ascending=False)

21 unnamed functions, 65 lines


,language,file,callbacks,lines
9,java,src/test/java/org/springframework/samples/petclinic/system/I18nPropertiesSyncTest.java,6,6
0,java,src/main/java/org/springframework/samples/petclinic/PetClinicRuntimeHints.java,3,3
1,java,src/main/java/org/springframework/samples/petclinic/owner/OwnerController.java,2,4
2,java,src/main/java/org/springframework/samples/petclinic/owner/PetController.java,2,4
5,java,src/test/java/org/springframework/samples/petclinic/PetClinicConcurrencyTests.java,2,37
7,java,src/test/java/org/springframework/samples/petclinic/service/ClinicServiceTests.java,2,4
3,java,src/main/java/org/springframework/samples/petclinic/owner/VisitController.java,1,2
4,java,src/main/java/org/springframework/samples/petclinic/system/CacheConfiguration.java,1,1
6,java,src/test/java/org/springframework/samples/petclinic/owner/PetTypeFormatterTests.java,1,3
8,java,src/test/java/org/springframework/samples/petclinic/system/CrashControllerTests.java,1,1


## 4. Sheet 2 — per language

In [16]:
sheet2 = C.parsers_table(APP, rows, matched, RUN, REFERENCES)
sheet2

,Language,Package,Repo,Files tried,Files parsed,Functions expected,Functions found,Evidence,Notes
0,bash,code-review-graph 2.3.8 (grammar: bash),spring-petclinic,2,2,NaN,NaN,results/spring-petclinic/runs/iter-002_2026-11-17_2008/coverage.ipynb,no reference parser for this language yet
1,java,code-review-graph 2.3.8 (grammar: java),spring-petclinic,50,50,195.0,192.0,results/spring-petclinic/runs/iter-002_2026-11-17_2008/coverage.ipynb,"expected from javac parser (JDK, java-ref); lost to same-name overwrite: 3; not found: 0; unnamed functions not coun..."
2,spring_config,code-review-graph 2.3.8 (grammar: spring_config),spring-petclinic,3,3,NaN,NaN,results/spring-petclinic/runs/iter-002_2026-11-17_2008/coverage.ipynb,no reference parser for this language yet
3,sql,code-review-graph 2.3.8 (grammar: sql),spring-petclinic,7,7,NaN,NaN,results/spring-petclinic/runs/iter-002_2026-11-17_2008/coverage.ipynb,no reference parser for this language yet
4,yaml,code-review-graph 2.3.8 (grammar: yaml),spring-petclinic,8,0,NaN,NaN,results/spring-petclinic/runs/iter-002_2026-11-17_2008/coverage.ipynb,no reference parser for this language yet


## 5. Sheet 3 — summary (same formulas as the workbook)

In [17]:
s = sheet1
summary = {'Files': len(s), 'Fully read': (s.read == 'Fully read').sum(), 'Partly read': (s.read == 'Partly read').sum(),
  'Not read': (s.read == 'Not read').sum(), 'Lines': int(s.lines.fillna(0).sum()), 'Lines read': int(s.lines_read.fillna(0).sum()),
  'Not shown to user': ((s.read != 'Fully read') & (s.shown == 'No')).sum()}
summary['% lines read'] = round(100 * summary['Lines read'] / summary['Lines'], 1)
summary

{'Files': 70,
 'Fully read': np.int64(61),
 'Partly read': np.int64(1),
 'Not read': np.int64(8),
 'Lines': 5487,
 'Lines read': 5226,
 'Not shown to user': np.int64(9),
 '% lines read': 95.2}

In [18]:
s[s.read == 'Not read'].why.value_counts()

why
Language dropped    8
Name: count, dtype: int64

## 6. Write the workbook

In [19]:
C.write_workbook(APP, rows, sheet2, OUT)

/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


PosixPath('/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/spring-petclinic/runs/iter-002_2026-11-17_2008/Ingestion_Coverage_spring-petclinic.xlsx')

## 7. Add this run to the repo's run log
`results/<repo>/runs.csv`, append-only: one line per run (per language with a reference parser).

In [20]:
log = C.log_run(APP, RUN, stamp, summary, sheet2)
pd.read_csv(log)

,run,run_at,app_commit,tool,files,fully_read,partly_read,not_read,lines,lines_read,pct_lines_read,language,functions_expected,functions_found,notes
0,iter-002_2026-11-17_2008,2026-11-17T20:08:21,500158f,code-review-graph 2.3.8,70,61,1,8,5487,5226,95.2,bash,NaN,NaN,no reference parser for this language yet
1,iter-002_2026-11-17_2008,2026-11-17T20:08:21,500158f,code-review-graph 2.3.8,70,61,1,8,5487,5226,95.2,java,195.0,192.0,NaN
2,iter-002_2026-11-17_2008,2026-11-17T20:08:21,500158f,code-review-graph 2.3.8,70,61,1,8,5487,5226,95.2,spring_config,NaN,NaN,no reference parser for this language yet
3,iter-002_2026-11-17_2008,2026-11-17T20:08:21,500158f,code-review-graph 2.3.8,70,61,1,8,5487,5226,95.2,sql,NaN,NaN,no reference parser for this language yet
4,iter-002_2026-11-17_2008,2026-11-17T20:08:21,500158f,code-review-graph 2.3.8,70,61,1,8,5487,5226,95.2,yaml,NaN,NaN,no reference parser for this language yet
